In [1]:
import os
import pandas as pd
from slugify import slugify
from corrosions.utils import worksheets, get_basename
import numpy as np

In [2]:
acvg_file = r"D:\Data\Rekap ACVG DCVG IDDA 2025_.xlsx"
excel_dir = r"D:\Projects\corrosions\tests\acvg_dcvg"

In [3]:
numeric_columns = [
    '% drop PCM',
    'On Potential (volt)',
    'Off Potential (volt)',
    'Latitude',
    'Longitude',
    'IR Drop (%)',
    'Kedalaman Pipa (m)',
    'Hasil ACVG (dB)',
]

In [4]:
def validate_numeric(value) -> float | None:
    if isinstance(value, float):
        return value
    if isinstance(value, str):
        return np.nan
    if pd.isna(value):
        return np.nan
    return float(value)

In [5]:
def get_province(area):
    if area == 'Tangerang' or area == 'Cilegon':
        return '36'
    if area == 'Jakarta':
        return '31'
    return '32'

In [6]:
def segment_code(row, sheet):
    return {
        'area_code': slugify(f"{sheet}-2025"),
        'name' : row['Segmen'],
        'diameter' : row['Diameter Pipa (inch)'],
        'code': slugify(f"{row['Segmen']}-{row['Diameter Pipa (inch)']}"),
        'province_code': get_province(sheet)
    }

In [7]:
sheets = worksheets(acvg_file)

In [8]:
segments = []
dfs = []

for sheet in sheets:
    selected_columns = [
        'Segmen',
        'Diameter Pipa (inch)',
        'Lokasi Anomali',
        'Kondisi Permukaan',
        '% drop PCM',
        'On Potential (volt)',
        'Off Potential (volt)',
        'DCVG',
        'ACVG',
        'Latitude',
        'Longitude',
        'IR Drop (%)',
        'Kedalaman Pipa (m)',
        'Hasil ACVG (dB)',
        'Keterangan',
    ]

    df = pd.read_excel(acvg_file, sheet_name=sheet)
    df = df[selected_columns]

    for numeric_column in numeric_columns:
        df.loc[:, numeric_column] = df[numeric_column].apply(validate_numeric)

    df['segment_code'] = df.apply(lambda row: slugify(f"{row['Segmen']} {row['Diameter Pipa (inch)']}"), axis=1)

    # Generate Segment JSON
    segment_uniques = df[['Segmen', 'Diameter Pipa (inch)']].drop_duplicates(keep='last')
    _segments = segment_uniques.apply(lambda row: segment_code(row, sheet), axis=1)
    segments = segments+(list(_segments.values))

    # Reorder Columns
    selected_columns.insert(0, 'segment_code')
    df = df.loc[:, list(selected_columns)]

    # Drop Columns
    df.drop(columns=['Segmen'], inplace=True)

    ## Renaming Columns
    df.rename(columns={
        'Diameter Pipa (inch)': 'diameter',
        'Lokasi Anomali': 'anomaly_location',
        'Kondisi Permukaan': 'surface_condition',
        '% drop PCM': 'drop_pcm',
        'On Potential (volt)': 'on_potential',
        'Off Potential (volt)': 'off_potential',
        'DCVG': 'survey_dcvg',
        'ACVG': 'survey_acvg',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'IR Drop (%)': 'ir_drop',
        'Kedalaman Pipa (m)': 'pipe_depth',
        'Hasil ACVG (dB)': 'result_acvg',
        'Keterangan': 'comment',
    }, inplace=True)

    # Save Area
    # _basename = slugify(get_basename(
    #     acvg_file,
    #     sheet,
    #     prefix="acvg_dcvg",
    # ))
    #
    # excel_filepath = os.path.join(
    #     excel_dir, f"{_basename}.xlsx"
    # )

    # df.to_excel(excel_filepath, index=False)
    # print(f"Saved to: {excel_filepath}")

    for _, segment in _segments.items():
        _df = df[df['segment_code'] == segment['code']]

        if len(_df) == 0:
            print(f"Kosong : {segment['code']}")

        # Save per Segment
        _basename = slugify(get_basename(
            segment['code'],
            sheet,
            prefix="acvg_dcvg",
        ))

        output_dir = os.path.join(excel_dir, slugify(sheet))
        os.makedirs(output_dir, exist_ok=True)

        excel_filepath = os.path.join(
            output_dir, f"{_basename}.xlsx"
        )

        _df.to_excel(excel_filepath, index=False)
        print(f"Saved to: {excel_filepath}")

    dfs.append(df)


Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-pertigaan-pln-bayur-end-cap-kawasan-akong-10-tangerang.xlsx
Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-looping-karawaci-jalan-thamrin-8-tangerang.xlsx
Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-pertigaan-olex-depan-pt-sinar-timur-6-tangerang.xlsx
Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-pertigaan-arah-ke-pt-lautan-rezeki-depan-pt-lautan-rezeki-6-tangerang.xlsx
Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-lampu-merah-perempatan-subandi-end-cap-karawaci-6-tangerang.xlsx
Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-pipa-servis-pt-lucky-indah-keramik-4-tangerang.xlsx
Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-pertigaan-rs-asobirin-depan-pt-indorama-ventures-indonesia-10-tangerang.xlsx
Saved to: D:\Projects\corrosions\tests\acvg_dcvg\tangerang\acvg-dcvg-pt-indorama-ventur

In [9]:
new_df = pd.concat(dfs, ignore_index=True)

In [10]:
new_df

,segment_code,diameter,anomaly_location,surface_condition,drop_pcm,on_potential,off_potential,survey_dcvg,survey_acvg,latitude,longitude,ir_drop,pipe_depth,result_acvg,comment
0,pertigaan-pln-bayur-end-cap-kawasan-akong-10,10,175 m dari simpang Jl Arya Kemuning,tanah,NaN,-1.057,NaN,2025-10-06,2025-08-10,-6.15441,106.60614,29.95,1.39,51,NaN
1,pertigaan-pln-bayur-end-cap-kawasan-akong-10,10,350 m dari simpang Jl Arya Kemuning,tanah,NaN,-1.057,NaN,2025-10-06,2025-08-10,-6.15336,106.60508,NaN,1.59,64,NaN
2,pertigaan-pln-bayur-end-cap-kawasan-akong-10,10,sekitar depan Arya Elang Emas,tanah,NaN,-1.008,NaN,2025-10-06,2025-08-10,-6.15122,106.60302,NaN,1.87,57,NaN
3,pertigaan-pln-bayur-end-cap-kawasan-akong-10,10,sekitar depan Pergudangan 38,tanah,NaN,-1.000,NaN,2025-10-06,2025-08-10,-6.15098,106.60275,NaN,1.56,52,NaN
4,pertigaan-pln-bayur-end-cap-kawasan-akong-10,10,sekitar depan PT.SCG Readymix,tanah,NaN,-0.974,NaN,2025-10-06,2025-08-10,-6.15056,106.60235,NaN,1.36,50,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,pipa-4-std-kedawung-bv-depan-mrs-sektor-a-b-4,4,NaN,Trotoar Beton,NaN,NaN,NaN,2025-10-22,2025-10-20,-6.71107,108.55570,NaN,0.91,55,NaN
398,pipa-4-std-kedawung-bv-depan-mrs-sektor-a-b-4,4,NaN,Trotoar Beton,NaN,NaN,NaN,2025-10-22,2025-10-20,-6.71095,108.55620,NaN,0.59,51,NaN
399,pipa-4-std-kedawung-bv-depan-mrs-sektor-a-b-4,4,NaN,Trotoar Beton,NaN,NaN,NaN,2025-10-22,2025-10-20,-6.70991,108.56072,NaN,0.62,49,NaN
400,kws-kiec-perempatan-pt-karunia-berca-indonesia...,8,Samping PT Karunia Berca,Tanah,NaN,NaN,NaN,2025-11-12,2025-11-05,-6.00574,106.01912,0.81,1.43,42,NaN
